# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install --quiet mlcroissant

## 1. Data Loading
Load metadata and records from the FAIR² dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load dataset
dataset = mlc.Dataset(croissant_url)

# Show basic dataset metadata
print("Name:", dataset.metadata.name)
print("Description:", dataset.metadata.description)
print("Version:", dataset.metadata.version)
print("Published:", dataset.metadata.datePublished)
print("License:", dataset.metadata.license)
print("Keywords:", getattr(dataset.metadata, 'keywords', ''))

## 2. Data Overview
Review available record sets, fields, and their `@id`. Listing all top-level data entities for exploration.

Each record set is referenced by its `@id`. Below, we discover and summarize them.

In [ ]:
# List all record sets and their fields
print("Available record sets:")
record_sets = list(dataset.metadata.recordSet)
for i, rs in enumerate(record_sets):
    print(f"[{i}] RecordSet @id: {rs['@id']}")
    fields = rs.get('field', [])
    if not isinstance(fields, list):
        fields = [fields]
    for field in fields:
        print(f"    - Field @id: {field['@id']}  label: {field.get('name', '')}")

## 3. Data Extraction
Load data from each record set into a dataframe for inspection and analysis, referencing each record set by its `@id`.

In [ ]:
# Extract each RecordSet's @id
record_set_ids = [rs['@id'] for rs in record_sets]

dataframes = {}
for rs_id in record_set_ids:
    print(f"Loading records from RecordSet: {rs_id}")
    records = list(dataset.records(record_set=rs_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"  - Loaded {len(df)} records. Columns: {df.columns.tolist()}")
    else:
        print(f"  - No records found.")

# For demonstration, print the first record set and its DataFrame
if record_set_ids:
    first_id = record_set_ids[0]
    if first_id in dataframes:
        print(f"\nFirst rows of RecordSet {first_id}:")
        display(dataframes[first_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply data processing such as filtering, normalization, and grouping using field `@id`s.

First, we select a numeric field from one record set (for illustration, we choose the first record set and pick the first numeric field, if available).

In [ ]:
import numpy as np

# Pick a record set with data
use_record_set_id = None
for rs_id in record_set_ids:
    if rs_id in dataframes:
        use_record_set_id = rs_id
        break

# Find a numeric field automatically
numeric_field = None
group_field = None
df = None
if use_record_set_id:
    df = dataframes[use_record_set_id]
    # Try to find a numeric column
    numeric_candidates = df.select_dtypes(include=[np.number]).columns.tolist()
    if numeric_candidates:
        numeric_field = numeric_candidates[0]
        # If other columns exist, use them as group, e.g., if a string column is present
        string_candidates = df.select_dtypes(include=[object]).columns.tolist()
        group_field = string_candidates[0] if string_candidates else None
        print(f"Selected numeric field for analysis: '{numeric_field}'")
        print(f"Selected group field for grouping: '{group_field}'")

if df is not None and numeric_field:
    # Example: filter values greater than threshold
    threshold = df[numeric_field].mean()   # as an example threshold
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Filtered {len(filtered_df)}/{len(df)} records with {numeric_field} > {threshold:.2f}")
    
    # Normalize the numeric column
    filtered_df[f"{numeric_field}_normalized"] = (
        (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    )
    print(f"First 5 normalized values:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
    
    # If a group field exists, group by it
    if group_field and group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index().sort_values(numeric_field, ascending=False)
        print(f"Mean of {numeric_field} by '{group_field}':")
        display(grouped_df.head())
else:
    print("No suitable numeric fields available for analysis.")

## 5. Visualization
Visualize the distribution of the selected numeric field and its normalized version (if available).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if df is not None and numeric_field:
    plt.figure(figsize=(10, 4))
    plt.subplot(1,2,1)
    sns.histplot(df[numeric_field].dropna(), kde=True, bins=30, color='skyblue')
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)

    if f"{numeric_field}_normalized" in filtered_df.columns:
        plt.subplot(1,2,2)
        sns.histplot(filtered_df[f"{numeric_field}_normalized"].dropna(), kde=True, bins=30, color='salmon')
        plt.title(f"Normalized {numeric_field} (filtered)")
        plt.xlabel(f"{numeric_field}_normalized")

    plt.tight_layout()
    plt.show()
    
    # If group_field exists, show a barplot
    if group_field and group_field in filtered_df.columns:
        plt.figure(figsize=(8,4))
        top_groups = filtered_df[group_field].value_counts().index[:10]
        plot_df = filtered_df[filtered_df[group_field].isin(top_groups)]
        sns.barplot(data=plot_df, x=group_field, y=numeric_field)
        plt.title(f"{numeric_field} by {group_field} (Top 10)")
        plt.xticks(rotation=30, ha='right')
        plt.show()

## 6. Conclusion
In this notebook, we've loaded and inspected the FAIR² dataset on mass spectrometry analysis of extracellular vesicles.

- We used `mlcroissant` to load the dataset by schema URL.
- All data entities, including record sets and fields, are referenced via their `@id`, enabling precise and reusable access paths.
- We explored numeric fields, filtered and normalized data, and visualized distributions to aid further biological or statistical analysis.

**Next steps:**
- Perform domain-specific data cleaning as necessary.
- Explore relationships between additional fields or across record sets.
- Apply ML and advanced statistical analysis as required by your research.